# 1 Importing and Merging Dataset

In [2]:
import pandas as pd
from fitparse import FitFile
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta

In [3]:
curr_date = "2026-06-03"

data_dir = Path(f"../data/{curr_date}")

wellness_files = sorted(data_dir.glob("*_WELLNESS.fit"))

print(f"Found {len(wellness_files)} WELLNESS files\n")

summary = []

for file in wellness_files:

    fitfile = FitFile(str(file))

    stress_count = 0

    for _ in fitfile.get_messages("stress_level"):
        stress_count += 1

    summary.append({
        "file": file.name,
        "stress_records": stress_count
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

Found 7 WELLNESS files



,file,stress_records
0,444927859217_WELLNESS.fit,477
1,444930358648_WELLNESS.fit,5
2,444931351296_WELLNESS.fit,4
3,444984229414_WELLNESS.fit,219
4,445043193231_WELLNESS.fit,231
5,445097413482_WELLNESS.fit,229
6,445149205969_WELLNESS.fit,196


In [4]:
#msg_counts = Counter(msg.name for msg in fitfile.get_messages())
#
#for msg_name, count in sorted(msg_counts.items(), key=lambda x: x[1], reverse=True):
#    print(f"{msg_name:25} {count}")

### Setting parameters

In [6]:
LOCAL_OFFSET = pd.Timedelta(hours=2)

### Stress

In [8]:
all_rows = []

for file in wellness_files:

    fitfile = FitFile(str(file))

    for msg in fitfile.get_messages("stress_level"):

        row = msg.get_values()

        all_rows.append({
            "source_file": file.name,
            "timestamp": row["stress_level_time"] + LOCAL_OFFSET,
            "stress": row["stress_level_value"],
            "body_battery": row.get("body_battery")
        })

stress_df = pd.DataFrame(all_rows)

# Only -1 is missing
stress_df["stress"] = stress_df["stress"].replace(-1, pd.NA)

# Sort chronologically
stress_df = stress_df.sort_values("timestamp")

# Remove duplicate timestamps coming from overlapping WELLNESS files
stress_df = (
    stress_df
    .drop_duplicates(subset="timestamp", keep="last")
    .reset_index(drop=True)
)

stress_df = stress_df[
    stress_df["timestamp"].dt.date
    == pd.to_datetime(curr_date).date()
].copy()

print("Rows:", len(stress_df))
print("Start:", stress_df["timestamp"].min())
print("End:", stress_df["timestamp"].max())

display(stress_df.head())
display(stress_df.tail())

Rows: 1360
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,stress,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,12,None
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,20,None
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,22,None
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,39,None
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,16,None


,source_file,timestamp,stress,body_battery
1355,445149205969_WELLNESS.fit,2026-06-03 23:55:00,17,None
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15,None
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,13,None
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,17,None
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,24,None


In [9]:
print("\nRows per source file:")
print(stress_df["source_file"].value_counts())

print("\nDuplicate timestamps:")
print(stress_df["timestamp"].duplicated().sum())


Rows per source file:
source_file
444927859217_WELLNESS.fit    477
445043193231_WELLNESS.fit    231
445097413482_WELLNESS.fit    229
444984229414_WELLNESS.fit    219
445149205969_WELLNESS.fit    195
444930358648_WELLNESS.fit      5
444931351296_WELLNESS.fit      4
Name: count, dtype: int64

Duplicate timestamps:
0


In [10]:
print(stress_df["timestamp"].min())
print(stress_df["timestamp"].max())

print(type(stress_df["timestamp"].iloc[0]))
print(stress_df["timestamp"].iloc[0].tzinfo)

2026-06-03 00:01:00
2026-06-03 23:59:00
<class 'pandas._libs.tslibs.timestamps.Timestamp'>
None


In [11]:
stress_df["timestamp"].dt.date.value_counts().sort_index()

timestamp
2026-06-03    1360
Name: count, dtype: int64

### Heart Rate

In [13]:
rows = []

target_date = pd.to_datetime(curr_date).date()

for file in wellness_files:

    fitfile = FitFile(str(file))

    for msg in fitfile.get_messages("monitoring"):

        row = msg.get_values()

        if "heart_rate" in row and "timestamp_16" in row:

            timestamp = (
                datetime.combine(
                    target_date,
                    datetime.min.time()
                )
                + timedelta(seconds=row["timestamp_16"])
            )

            rows.append({
                "source_file": file.name,
                "timestamp": timestamp,
                "heart_rate": row["heart_rate"]
            })

heart_rate_df = (
    pd.DataFrame(rows)
    .sort_values("timestamp")
    .drop_duplicates(subset="timestamp", keep="last")
    .reset_index(drop=True)
)

print("Rows:", len(heart_rate_df))
print("Start:", heart_rate_df["timestamp"].min())
print("End:", heart_rate_df["timestamp"].max())

display(heart_rate_df.head())
display(heart_rate_df.tail())

Rows: 1239
Start: 2026-06-03 00:00:56
End: 2026-06-03 18:12:12


,source_file,timestamp,heart_rate
0,445097413482_WELLNESS.fit,2026-06-03 00:00:56,72
1,445097413482_WELLNESS.fit,2026-06-03 00:01:56,71
2,445097413482_WELLNESS.fit,2026-06-03 00:02:56,74
3,445097413482_WELLNESS.fit,2026-06-03 00:03:56,73
4,445097413482_WELLNESS.fit,2026-06-03 00:04:56,80


,source_file,timestamp,heart_rate
1234,445097413482_WELLNESS.fit,2026-06-03 18:07:12,68
1235,445097413482_WELLNESS.fit,2026-06-03 18:09:12,76
1236,445097413482_WELLNESS.fit,2026-06-03 18:10:12,73
1237,445097413482_WELLNESS.fit,2026-06-03 18:11:12,72
1238,445097413482_WELLNESS.fit,2026-06-03 18:12:12,70


### Body Battery

In [15]:
all_rows = []

for file in wellness_files:

    fitfile = FitFile(str(file))

    for msg in fitfile.get_messages("stress_level"):

        row = msg.get_values()

        all_rows.append({
            "source_file": file.name,
            "timestamp": row["stress_level_time"] + LOCAL_OFFSET,
            "body_battery": row["unknown_3"]
        })

body_battery_df = pd.DataFrame(all_rows)

body_battery_df["body_battery"] = (
    body_battery_df["body_battery"]
    .replace(-1, pd.NA)
)

body_battery_df = (
    body_battery_df
    .sort_values("timestamp")
    .drop_duplicates(subset="timestamp", keep="last")
    .reset_index(drop=True)
)

print("Rows:", len(body_battery_df))
print("Start:", body_battery_df["timestamp"].min())
print("End:", body_battery_df["timestamp"].max())

body_battery_df.head()

Rows: 1361
Start: 2026-06-03 00:01:00
End: 2026-06-04 00:00:00


,source_file,timestamp,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,32
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,32
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,32
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,32
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,32


In [16]:
body_battery_df.tail()

,source_file,timestamp,body_battery
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,21
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,23
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,23
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,23
1360,445149205969_WELLNESS.fit,2026-06-04 00:00:00,23


### Respirational Rate

In [18]:
GARMIN_EPOCH = datetime(1989, 12, 31)

all_rows = []

for file in wellness_files:

    fitfile = FitFile(str(file))

    for msg in fitfile.get_messages("unknown_297"):

        row = msg.get_values()

        all_rows.append({
            "source_file": file.name,
            "timestamp": (
                GARMIN_EPOCH
                + timedelta(seconds=row["unknown_253"])
                + LOCAL_OFFSET
            ),
            "respiration_rate": row["unknown_0"] / 100
        })

respiration_df = pd.DataFrame(all_rows)

respiration_df.loc[
    respiration_df["respiration_rate"] < 0,
    "respiration_rate"
] = pd.NA

respiration_df = (
    respiration_df
    .sort_values("timestamp")
    .drop_duplicates(subset="timestamp", keep="last")
    .reset_index(drop=True)
)

print("Rows:", len(respiration_df))
print("Start:", respiration_df["timestamp"].min())
print("End:", respiration_df["timestamp"].max())

respiration_df.head()

Rows: 1440
Start: 2026-06-03 00:01:00
End: 2026-06-04 00:00:00


,source_file,timestamp,respiration_rate
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,13.58
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,15.36
2,444927859217_WELLNESS.fit,2026-06-03 00:03:00,14.91
3,444927859217_WELLNESS.fit,2026-06-03 00:04:00,16.58
4,444927859217_WELLNESS.fit,2026-06-03 00:05:00,NaN


In [19]:
respiration_df.tail()

,source_file,timestamp,respiration_rate
1435,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15.25
1436,445149205969_WELLNESS.fit,2026-06-03 23:57:00,15.00
1437,445149205969_WELLNESS.fit,2026-06-03 23:58:00,14.33
1438,445149205969_WELLNESS.fit,2026-06-03 23:59:00,15.16
1439,445149205969_WELLNESS.fit,2026-06-04 00:00:00,15.08


### Merging

In [21]:
# ============================================================
# Build unified 1-minute dataframe for the selected day
# ============================================================

# Ensure datetime dtype
for df in [stress_df, body_battery_df, respiration_df, heart_rate_df]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

# Keep only records from the target day
target_date = pd.to_datetime(curr_date).date()

stress_df = stress_df[
    stress_df["timestamp"].dt.date == target_date
].copy()

body_battery_df = body_battery_df[
    body_battery_df["timestamp"].dt.date == target_date
].copy()

respiration_df = respiration_df[
    respiration_df["timestamp"].dt.date == target_date
].copy()

heart_rate_df = heart_rate_df[
    heart_rate_df["timestamp"].dt.date == target_date
].copy()

# Full minute timeline for the day
full_index = pd.date_range(
    start=pd.Timestamp(curr_date),
    periods=1440,
    freq="1min"
)

# ------------------------------------------------------------
# Convert any dataframe to 1-minute resolution
# ------------------------------------------------------------

def prepare_minute_df(df, value_col):

    temp = df.copy()

    # Round timestamps down to minute
    temp["timestamp"] = temp["timestamp"].dt.floor("min")

    # If multiple values exist within a minute, average them
    temp = (
        temp.groupby("timestamp")[value_col]
        .mean()
        .to_frame()
    )

    return temp


stress_min = prepare_minute_df(
    stress_df,
    "stress"
)

body_battery_min = prepare_minute_df(
    body_battery_df,
    "body_battery"
)

respiration_min = prepare_minute_df(
    respiration_df,
    "respiration_rate"
)

heart_rate_min = prepare_minute_df(
    heart_rate_df,
    "heart_rate"
)

# ------------------------------------------------------------
# Merge everything into one daily dataframe
# ------------------------------------------------------------

daily_df = pd.DataFrame(index=full_index)

daily_df = daily_df.join(stress_min)
daily_df = daily_df.join(body_battery_min)
daily_df = daily_df.join(respiration_min)
daily_df = daily_df.join(heart_rate_min)

daily_df.index.name = "timestamp"

# Move timestamp back to a normal column
daily_df = daily_df.reset_index()

# ------------------------------------------------------------
# Quick checks
# ------------------------------------------------------------

print("Shape:", daily_df.shape)

print("\nMissing values:")
print(daily_df.isna().sum())

print("\nDate range:")
print(daily_df["timestamp"].min())
print(daily_df["timestamp"].max())

display(daily_df.head(20))

Shape: (1440, 5)

Missing values:
timestamp             0
stress              221
body_battery         80
respiration_rate    307
heart_rate          446
dtype: int64

Date range:
2026-06-03 00:00:00
2026-06-03 23:59:00


,timestamp,stress,body_battery,respiration_rate,heart_rate
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0
1,2026-06-03 00:01:00,12.0,32.0,13.58,71.0
2,2026-06-03 00:02:00,20.0,32.0,15.36,74.0
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0
4,2026-06-03 00:04:00,22.0,32.0,16.58,80.0
5,2026-06-03 00:05:00,39.0,32.0,NaN,76.0
6,2026-06-03 00:06:00,NaN,NaN,15.25,69.0
7,2026-06-03 00:07:00,16.0,32.0,13.75,71.0
8,2026-06-03 00:08:00,18.0,32.0,15.08,72.0
9,2026-06-03 00:09:00,19.0,32.0,14.75,76.0


In [22]:
print(stress_df.shape)
print(body_battery_df.shape)
print(respiration_df.shape)
print(heart_rate_df.shape)

(1360, 4)
(1360, 3)
(1439, 3)
(1239, 3)


In [23]:
(stress_df["stress"] == -1).sum()

np.int64(0)

In [24]:
(stress_df["stress"] == -2).sum()

np.int64(228)

In [25]:
stress_df[
    stress_df["stress"].isna()
][["timestamp", "stress"]].head(20)

,timestamp,stress
61,2026-06-03 01:07:00,<NA>
62,2026-06-03 01:08:00,<NA>
63,2026-06-03 01:09:00,<NA>
64,2026-06-03 01:10:00,<NA>
68,2026-06-03 01:15:00,<NA>
69,2026-06-03 01:16:00,<NA>
70,2026-06-03 01:17:00,<NA>
71,2026-06-03 01:18:00,<NA>
72,2026-06-03 01:19:00,<NA>
73,2026-06-03 01:20:00,<NA>


# 2 EDA

### Missing Values

In [28]:
daily_df.isna().sum()

timestamp             0
stress              221
body_battery         80
respiration_rate    307
heart_rate          446
dtype: int64